# Spam SMS Detection

This notebook walks through the end-to-end process for detecting spam SMS messages using the SMS Spam Collection dataset. It includes data loading, EDA, preprocessing, feature engineering, model training, evaluation, and saving the best model.

## 1. Import Libraries

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

nltk.download('stopwords', quiet=True)
sns.set_style('whitegrid')

## 2. Load the Dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'spam.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Shape:', df.shape)
print('Missing values:
', df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())
df['label'].value_counts()

In [ ]:
df['message_length'] = df['message'].apply(lambda text: len(str(text)))
df['message_word_count'] = df['message'].apply(lambda text: len(str(text).split()))
df[['message_length', 'message_word_count']].describe()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x='label', data=df, palette=['#4CAF50', '#F44336'])
plt.title('Class Distribution')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(6, 6))
df['label'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=140, textprops={'color':'black'})
plt.ylabel('')
plt.title('Spam vs Ham Share')
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(df['message_length'], bins=20, kde=True, color='#2196F3')
plt.title('Message Length Distribution')
plt.xlabel('Message Length (characters)')
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(x='label', y='message_length', data=df, palette=['#4CAF50', '#F44336'])
plt.title('Message Length by Label')
plt.xlabel('Label')
plt.ylabel('Message Length')
plt.show()

## 4. Data Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_message(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z]', ' ', text)
    tokens = text.split()
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [stemmer.stem(token) for token in tokens]
    return ' '.join(tokens)

df['clean_message'] = df['message'].apply(preprocess_message)
df.head()

## 5. Feature Engineering

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['clean_message'])
y = df['label'].map({'ham': 0, 'spam': 1})
X.shape

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

## 7. Train Models

In [ ]:
models = {
    'MultinomialNB': MultinomialNB(),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'LinearSVC': LinearSVC(random_state=42, max_iter=10000),
}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds),
        'recall': recall_score(y_test, preds),
        'f1_score': f1_score(y_test, preds),
        'confusion_matrix': confusion_matrix(y_test, preds),
        'classification_report': classification_report(y_test, preds, target_names=['Ham', 'Spam']),
    })

results_df = pd.DataFrame(results)[['model', 'accuracy', 'precision', 'recall', 'f1_score']]
results_df

## 8. Select Best Model and Save Artifacts

In [ ]:
best_result = max(results, key=lambda item: item['f1_score'])
best_model_name = best_result['model']
best_model_name

best_model = models[best_model_name]

In [ ]:
import joblib
artifact_dir = os.path.join('..', 'models')
os.makedirs(artifact_dir, exist_ok=True)
joblib.dump(best_model, os.path.join(artifact_dir, 'spam_model.pkl'))
joblib.dump(vectorizer, os.path.join(artifact_dir, 'tfidf_vectorizer.pkl'))
print('Saved best model and TF-IDF vectorizer')

## 9. Prediction Function

In [ ]:
def predict_message(text):
    processed = preprocess_message(text)
    features = vectorizer.transform([processed])
    prediction = best_model.predict(features)[0]
    return 'Spam' if prediction == 1 else 'Ham'

sample_texts = [
    'Congratulations! You won a free lottery ticket.',
    'Hi, where are you?',
]
for text in sample_texts:
    print(f'Input: {text}')
    print('Prediction:', predict_message(text))
    print()